# 05 - Cypher Query Generation

Orthograph can generate Cypher queries from your model definitions, similar to how an ORM generates SQL. This ensures that your queries are consistent with your model and reduces boilerplate when working with Neo4j or other Cypher-compatible databases.

This notebook covers:
- Creating a `CypherGenerator` from a `GraphDataModel`
- Generating MERGE and CREATE queries for nodes
- Generating relationship queries
- Generating uniqueness constraints
- Generating MATCH patterns
- A full workflow combining all operations

In [ ]:
from typing import Optional

from orthograph import (
    Cardinality,
    GraphDataModel,
    NodeModel,
    RelationshipModel,
)
from orthograph.extensions.cypher import CypherGenerator


# Define the filmography model
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int


class City(NodeModel):
    __label__ = "City"
    __uid_field__ = "name"
    name: str


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_type__ = Person
    __target_type__ = Movie
    role: str


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_type__ = Person
    __target_type__ = Movie


class LivesIn(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_type__ = Person
    __target_type__ = City
    __source_cardinality__ = Cardinality.ZERO_OR_ONE


model = GraphDataModel(
    name="Filmography",
    node_types=[Person, Movie, City],
    relationship_types=[ActedIn, Directed, LivesIn],
)

print("Model:", model.name)
print("Nodes:", model.node_labels)
print("Rels: ", model.relationship_labels)

## Creating the generator

The `CypherGenerator` takes a `GraphDataModel` and provides methods for generating Cypher queries.

In [ ]:
gen = CypherGenerator(model)
print("CypherGenerator ready for model:", gen.model.name)

## Generating MERGE queries

MERGE uses the `uid_field` to match an existing node. If the node exists, it updates the remaining properties via SET. If it does not exist, it creates it. This is the idempotent way to upsert nodes.

The data dict must include `__label__` and any properties to set.

In [ ]:
# MERGE a Person node
query, params = gen.merge_node(
    {
        "__label__": "Person",
        "name": "Keanu Reeves",
        "born": 1964,
    }
)
print("Query: ", query)
print("Params:", params)
print()

# MERGE a Movie node
query, params = gen.merge_node(
    {
        "__label__": "Movie",
        "title": "The Matrix",
        "year": 1999,
    }
)
print("Query: ", query)
print("Params:", params)

## Generating CREATE queries

Use `create_node` for unconditional creation -- when you know the node does not yet exist, or when you do not need idempotency.

In [ ]:
query, params = gen.create_node(
    {
        "__label__": "City",
        "name": "Los Angeles",
    }
)
print("Query: ", query)
print("Params:", params)

## Relationship queries

Both `create_relationship` and `merge_relationship` generate queries that first MATCH the source and target nodes by their UID fields, then CREATE or MERGE the relationship between them.

The data dict must include:
- `__label__` -- the relationship label
- `__source_uid__` -- the UID value of the source node
- `__target_uid__` -- the UID value of the target node
- Any additional keys are treated as relationship properties

In [ ]:
# CREATE a relationship with properties (ACTED_IN has a 'role' property)
query, params = gen.create_relationship(
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    }
)
print("CREATE relationship:")
print("  Query: ", query)
print("  Params:", params)
print()

# MERGE a relationship without properties (DIRECTED has no properties)
query, params = gen.merge_relationship(
    {
        "__label__": "DIRECTED",
        "__source_uid__": "Lana Wachowski",
        "__target_uid__": "The Matrix",
    }
)
print("MERGE relationship:")
print("  Query: ", query)
print("  Params:", params)

## Generating constraints

The generator can produce uniqueness constraint statements from the `uid_field` definitions in the model. These are typically run once when setting up the database schema.

In [ ]:
constraints = gen.generate_constraints()
print(f"Generated {len(constraints)} constraint(s):\n")
for c in constraints:
    print(c)

## MATCH patterns

Generate read queries to retrieve nodes of a given type, or to match a specific relationship pattern.

In [ ]:
# Match all Person nodes
print("Match Person:", gen.match_node(Person))
print()

# Match ACTED_IN relationship pattern
print("Match ACTED_IN:", gen.match_relationship(ActedIn))
print()

# Match LIVES_IN relationship pattern
print("Match LIVES_IN:", gen.match_relationship(LivesIn))

## Putting it together

A typical database setup workflow: generate constraints first, then merge nodes, then create relationships. The output below shows the full sequence of Cypher statements that would populate a small filmography database.

In [ ]:
print("=" * 60)
print("STEP 1: Create uniqueness constraints")
print("=" * 60)
for c in gen.generate_constraints():
    print(c)
print()

print("=" * 60)
print("STEP 2: Merge nodes")
print("=" * 60)
people = [
    {"__label__": "Person", "name": "Keanu Reeves", "born": 1964},
    {"__label__": "Person", "name": "Lana Wachowski", "born": 1965},
    {"__label__": "Person", "name": "Carrie-Anne Moss", "born": 1967},
]
movies = [
    {"__label__": "Movie", "title": "The Matrix", "year": 1999},
    {"__label__": "Movie", "title": "The Matrix Reloaded", "year": 2003},
]
cities = [
    {"__label__": "City", "name": "Los Angeles"},
]

for node_data in people + movies + cities:
    q, p = gen.merge_node(node_data)
    print(f"  {q}")
    print(f"    params: {p}")
print()

print("=" * 60)
print("STEP 3: Create relationships")
print("=" * 60)
relationships = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Carrie-Anne Moss",
        "__target_uid__": "The Matrix",
        "role": "Trinity",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix Reloaded",
        "role": "Neo",
    },
    {
        "__label__": "DIRECTED",
        "__source_uid__": "Lana Wachowski",
        "__target_uid__": "The Matrix",
    },
    {
        "__label__": "DIRECTED",
        "__source_uid__": "Lana Wachowski",
        "__target_uid__": "The Matrix Reloaded",
    },
    {
        "__label__": "LIVES_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "Los Angeles",
    },
]

for rel_data in relationships:
    q, p = gen.create_relationship(rel_data)
    print(f"  {q}")
    print(f"    params: {p}")

## Undirected Relationship Queries

When a relationship type has `__directed__ = False`, the generated Cypher uses `-`
instead of `->` for both MATCH and CREATE/MERGE patterns. This tells the query engine
that direction does not matter.

This works for both same-type endpoints (e.g. `Person`-`Person`) and cross-type endpoints
(e.g. `Person`-`Company`).

In [ ]:
# Add undirected relationships to the model
class Company(NodeModel):
    __label__ = "Company"
    __uid_field__ = "name"
    name: str


class FriendOf(RelationshipModel):
    __label__ = "FRIEND_OF"
    __source_type__ = Person
    __target_type__ = Person
    __directed__ = False
    since: Optional[int] = None


class Collaborates(RelationshipModel):
    __label__ = "COLLABORATES"
    __source_type__ = Person
    __target_type__ = Company
    __directed__ = False


extended_model = GraphDataModel(
    name="Extended",
    node_types=[Person, Movie, City, Company],
    relationship_types=[ActedIn, Directed, LivesIn, FriendOf, Collaborates],
)
gen2 = CypherGenerator(extended_model)
print("Extended model created with undirected relationships.")

In [ ]:
# MATCH pattern for undirected relationship -- uses '-' instead of '->'
print("Directed MATCH:")
print(" ", gen2.match_relationship(ActedIn))
print()
print("Undirected MATCH (same-type):")
print(" ", gen2.match_relationship(FriendOf))
print()
print("Undirected MATCH (cross-type):")
print(" ", gen2.match_relationship(Collaborates))

In [ ]:
# CREATE/MERGE for undirected relationships also use '-' instead of '->'
print("CREATE undirected relationship (with property):")
q, p = gen2.create_relationship(
    {
        "__label__": "FRIEND_OF",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "Carrie-Anne Moss",
        "since": 1999,
    }
)
print(f"  {q}")
print(f"  params: {p}")
print()

print("MERGE undirected cross-type relationship:")
q, p = gen2.merge_relationship(
    {
        "__label__": "COLLABORATES",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "Warner Bros",
    }
)
print(f"  {q}")
print(f"  params: {p}")
print()

# Compare with directed CREATE
print("CREATE directed relationship (for comparison):")
q, p = gen2.create_relationship(
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Keanu Reeves",
        "__target_uid__": "The Matrix",
        "role": "Neo",
    }
)
print(f"  {q}")
print(f"  params: {p}")